# Preamble

In [2]:
import os

import pandas as pd
import country_converter
import pycountry

# Prepare input for HRP

In [2]:
# load csv from ip2location file (easier as it is present in local machine...)

ip2loc_files = [
    "ip2location/output/nl-ens_20260204_netstats_ip2location.csv",
]

for ip2loc_file in ip2loc_files:
    hrp_input_file = os.path.basename(ip2loc_file).split(".")[0]
    hrp_input_file = f"{hrp_input_file}_hrp_input.csv"
    ip2loc_pdf = pd.read_csv(ip2loc_file)
    ip2loc_pdf["ip"].to_csv(f"../hrp/input/{hrp_input_file}", index=False)

In [ ]:
# example of execution from the "hrp" folder
# cd anycast-service-discovery/measurements/hrp
# cut -d, -f1 input/nl-ens_20260202_netstats_ip2location_hrp_input.csv | tail -n +2 | sort -n -t . -k 1,1 -k 2,2 -k 3,3 -k 4,4 -S10% -T /tmp --compress-program=lz4 | ../../venv/bin/python search-for-hrps.py --scan-port 80 --scan-date 20260125 -o output/ --pyasn-file ipasn_20260125.dat --scan-location nl-ens -
# gunzip output/slash24-info.csv.gz
# mv output/slash24-info.csv output/nl-ens_20260202_slash24-info.csv
# mc mv output/nl-ens_20260202_slash24-info.csv storage/luvizottocesarg-tmp/anycast-service-discovery/hrp/output/

# Prepare ccTLD list

In [4]:
# extracted from https://www.worldstandards.eu/other/tlds/
# used the tool from https://gist.github.com/derlin/421d2bb55018a1538271227ff6b1299d
# resulted in country_tld.csv
# extracted from https://www.iana.org/domains/root/db + adding Taiwan (tw)
# copied to a excell sheet; get punycodes with chatgpt
# resulted in cctlds_country.csv
country_tld_pdf = pd.read_csv("country_tld.csv")
country_tld_pdf.columns = ["country", "tld"]
tlds_pdf = country_tld_pdf.groupby("tld").size().reset_index(name="count").sort_values("count", ascending=False)
tlds_pdf["tld"] = tlds_pdf["tld"].str.replace('.', '')
tlds_pdf["tld"] = tlds_pdf["tld"].str.strip()
cctlds = set(tlds_pdf["tld"].to_list())

country_convert_mapping = {}
for cctld in cctlds:
    alpha_3 = None
    if pycountry.countries.get(alpha_2=cctld) is not None:
        alpha_3 = pycountry.countries.get(alpha_2=cctld).alpha_3
    else:
        alpha_3 = country_converter.convert(cctld)
    country_convert_mapping.update({cctld: alpha_3})

# https://www.iban.com/country-codes
country_convert_mapping["abudhabi"] = "ARE"  # abu dhabi
country_convert_mapping["gal"] = "ESP"  # galicia
country_convert_mapping["an"] = "NLD"  # netherlands antilles
country_convert_mapping["tp"] = "TLS"  # timor leste
country_convert_mapping["eus"] = "ESP"  # basque country
country_convert_mapping["ac"] = "GBR"  # Ascension Island
country_convert_mapping["cat"] = "ESP"  # Catalonia
country_convert_mapping["scot"] = "BGR"  # scotland is great britain
country_convert_mapping["swiss"] = country_converter.convert("swiss")  # CHE
missing = [tld for tld, alpha3 in country_convert_mapping.items() if alpha3 == "not found"]
print("to fix", len(missing), missing)
# limitation: .co can be Colombia but it can also be used in british websites

for _, row in pd.read_csv("cctlds_country.csv").iterrows():
    cctld = row["cctld"]
    if "xn--" in cctld:
        cctld = row["administered"]
        #print(cctld)
    if cctld == "eu":
        # Europe should not be added...
        continue
    if cctld == "su":
        # su is controlled by Russia according to IANA
        country_convert_mapping[cctld] = "RUS"
        continue

    # update the country_convert_mapping with administered countries (with or without punycode)
    if cctld not in country_convert_mapping.keys():
        cc1 = pycountry.countries.get(alpha_2=cctld)
        cc2 = country_converter.convert(cctld)
        if cc1 is None and cc2 == "not found":
            print("manually add:", cctld, row["country_full"])
        else:
            if cc1 is None:
                country_convert_mapping[cctld] = cc2
            else:
                country_convert_mapping[cctld] = cc1.alpha_3

    # now adding the punycode
    if "xn--" in row["cctld"]:
        punycode = row["cctld"]
        administered = row["administered"]
        alpha_3 = country_convert_mapping[administered]
        country_convert_mapping[punycode] = alpha_3


#for country in pycountry.countries:
#    cctlds.add(f"{country.alpha_2.lower()}")

#cctlds.remove("eu")  # European Union. Remove from our ccTLD list.
#print(len(cctlds))

# punycodes from IANA
country_convert_mapping["xn--p1ai"] = "RUS"
country_convert_mapping["xn--4dbrk0ce"] = "ISR"
country_convert_mapping["xn--o3cw4h"] = "THA"
country_convert_mapping["xn--j1amh"] = "UKR"
country_convert_mapping["xn--kpry57d"] = "TWN"
country_convert_mapping["xn--d1alf"] = "MKD"
country_convert_mapping["xn--ygbi2ammx"] = "PSE"  # palestine state
country_convert_mapping["xn--xkc2dl3a5ee0h"] = "IND"
country_convert_mapping["xn--ogbpf8fl"] = "SYR"
country_convert_mapping["xn--kprw13d"] = "TWN"
country_convert_mapping["xn--pgbs0dh"] = "TUN"
country_convert_mapping["xn--mgbah1a3hjkrd"] = "MRT"
country_convert_mapping["xn--h2breg3eve"] = "IND"
country_convert_mapping["xn--h2brj9c8c"] = "IND"
country_convert_mapping["xn--mgb9awbf"] = "OMN"
country_convert_mapping["xn--mgbai9azgqp6j"] = "PAK"
country_convert_mapping["xn--mgbbh1a"] = "IND"
country_convert_mapping["xn--mgbc0a9azcg"] = "MAR"
country_convert_mapping["xn--mgbgu82a"] = "IND"

# DELETE EU!!!
del country_convert_mapping["eu"]

country_convert_mapping["gov"] = "USA"  # gov is administered by US
country_convert_mapping["mil"] = "USA"  # mil is administered by US

print(len(country_convert_mapping.keys()))

#cctld_df = spark.createDataFrame([(cctld,) for cctld in cctlds], ["tld"])
#print(cctld_df.count())

iana_punycodes = pd.read_csv("iana_punycodes.csv")["cctld"].to_list()

p = dict({})
for iana_punycode in iana_punycodes:
    latin = iana_punycode.encode('idna').decode('ascii')
    if latin not in country_convert_mapping.keys():
        p[latin] = iana_punycode

for k,v in p.items():
    print(k, v)

cat not found in ISO3
tp not found in ISO2
abudhabi not found in regex
gal not found in ISO3
an not found in ISO2
eu not found in ISO2
ac not found in ISO2
eus not found in ISO3


to fix 1 ['eu']
324
xn--e1a4c ею
xn--qxa6a ευ


In [ ]:
pd.DataFrame(list(country_convert_mapping.items()), columns=["cctld", "alpha_3"]
).to_csv("cctld_alpha_3_mapping.csv", index=False, header=True)